# Desarrollo de Aplicaciones Web (DAB-2201)
## Sesión 09: Middleware Web, Intercepción y Logging de Transacciones
**Docente Titular:** MCC. Iván Márquez Larios (`ijmarquezl`)
**Repositorio Maestro:** `https://github.com/ijmarquezl/tecnm-desarrollo-web`

---
### Protocolo de la Sesión
1. **Fase 1 (El Artesano):** Resolver a mano: *Middleware manual de tiempo de respuesta (X-Process-Time)*.
2. **Fase 2 (El Científico):** Definir contrato C-R-E-O: *Interceptor de peticiones y auditoría de IP/User-Agent* y formular prompt.
3. **Fase 3 (El Auditor):** Someter el servidor al Test Harness: *Pruebas de inyección de encabezados de rastreo*.
4. **Entrega:** Documentar en `auditorias/auditoria_sesion09.md`.

### Paso 0: Inicialización del Entorno (FastAPI y PHP 8+)

In [1]:
import os
print('⏳ Instalando FastAPI, Uvicorn y HTTPX para pruebas...')
!pip install -q fastapi uvicorn httpx pydantic passlib python-jose[cryptography] sqlalchemy alembic
if os.system('which php > /dev/null 2>&1') != 0:
    print('⏳ Instalando PHP CLI...')
    !apt-get update -qq && apt-get install -y -qq php-cli > /dev/null 2>&1
print('✅ Entorno Web listo:')
!python3 --version
!php -v | head -n 1

⏳ Instalando FastAPI, Uvicorn y HTTPX para pruebas...
⏳ Instalando PHP CLI...
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
✅ Entorno Web listo:
Python 3.13.18
PHP 8.3.6 (cli) (built: Oct 19 2026 15:16:58) (NTS)


---
## SECCIÓN A: TRACK PYTHON / FASTAPI
### Fase 1: El Artesano (15 min - Sin IA)
**Reto:** Middleware manual de tiempo de respuesta (X-Process-Time)

In [2]:
%%writefile app_artesano.py
import time
from fastapi import FastAPI, HTTPException, status, Request
from pydantic import BaseModel

app = FastAPI(title="API Artesano - Sesión 09")

class PayloadRequest(BaseModel):
    dato: str


@app.post("/api/recurso")
def procesar_recurso(payload: PayloadRequest):
    if not payload.dato.strip():
        raise HTTPException(status_code=status.HTTP_400_BAD_REQUEST, detail="Dato vacio")
    return {"status": "success", "mensaje": f"Procesado: {payload.dato}"}


@app.middleware("http")
async def add_process_time_header(request: Request, call_next):
    # 1. Antes de que la petición llegue al endpoint, tomamos el tiempo inicial
    start_time = time.time()

    # 2. Dejamos que la petición
    response = await call_next(request)

    process_time = time.time() - start_time
    response.headers["X-Process-Time"] = str(process_time)

    return response


Writing app_artesano.py


### Fase 2: El Científico (Prompting C-R-E-O)
Pega aquí la implementación optimizada propuesta por la IA:

In [5]:
%%writefile app_ia.py
import time
from fastapi import FastAPI, Request
from pydantic import BaseModel

app = FastAPI(title="API Científico - Sesión 09")

class PayloadRequest(BaseModel):
    dato: str

@app.post("/api/recurso")
def procesar_recurso(payload: PayloadRequest):
    return {"status": "success", "mensaje": f"Procesado: {payload.dato}"}

# --- RETO DEL CIENTÍFICO: Interceptor de auditoría y rastreo ---
@app.middleware("http")
async def audit_and_trace_middleware(request: Request, call_next):
    start_time = time.time()

    # Extraemos datos del cliente para auditoría
    client_ip = request.client.host if request.client else "Desconocida"
    user_agent = request.headers.get("User-Agent", "Desconocido")

    # Dejamos pasar la petición
    response = await call_next(request)

    # Inyectamos múltiples headers de rastreo en la respuesta
    process_time = time.time() - start_time
    response.headers["X-Process-Time"] = str(process_time)
    response.headers["X-Client-IP"] = client_ip
    response.headers["X-User-Agent"] = user_agent
    response.headers["X-Transaction-ID"] = "tx-123456789" # Simulando un ID de transacción

    # Imprimimos en consola el log de la transacción
    print(f"[LOG] Ruta: {request.url.path} | IP: {client_ip} | Tiempo: {process_time:.4f}s")

    return response


Overwriting app_ia.py


### Fase 3: El Auditor (Test Harness HTTP)
Pruebas de servidor: Pruebas de inyección de encabezados de rastreo

In [3]:
import threading
import time
import uvicorn
import httpx
from app_artesano import app

def run_server():
    uvicorn.run(app, host="127.0.0.1", port=8000, log_level="warning")

server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()
time.sleep(1.5)

client = httpx.Client(base_url="http://127.0.0.1:8000")
print("=== AUDITORIA CRITICA WEB: SESION 09 ===\n")
r = client.post("/api/recurso", json={"dato": "Prueba Test Harness"})
print(f"Respuesta Servidor: HTTP {r.status_code} - {r.json()}")
assert r.status_code == 200
print("PASO ✅ - Test Harness HTTP ejecutado con exito.")


=== AUDITORIA CRITICA WEB: SESION 09 ===

Respuesta Servidor: HTTP 200 - {'status': 'success', 'mensaje': 'Procesado: Prueba Test Harness'}
PASO ✅ - Test Harness HTTP ejecutado con exito.
